# 강의 06 · 실습 9 — 패턴 3 핸드오프 · (3) 변형

## 1. 문제상황

- 인터넷 서비스 고객센터는 접수 담당, 기술 담당, 보상 담당으로 나뉘어 있습니다.
- 고객이 「사흘째 인터넷이 끊겨서 재택근무를 못 하고 있고, 보상도 받고 싶다」고 접수 담당에게 말하면, 접수 담당은 기술 담당 번호를 알려 주고 끝냅니다.
- 고객이 기술 담당에게 다시 전화해 장애 조치를 들으면, 기술 담당은 보상 문제는 보상 담당에게 물어보라고 합니다. 고객은 세 번째 전화를 겁니다.
- 담당이 바뀔 때마다 고객은 같은 이야기를 처음부터 다시 하고, 어느 담당을 거쳐 왔는지는 아무 데도 남지 않습니다.

## 2. 문제와 목표

- **문제**: 한 문의가 두 에이전트를 거쳐야 하는데, 에이전트가 바뀔 때마다 고객이 다시 문의해야 하고 대화 이력이 끊깁니다. 어느 에이전트를 거쳤는지도 기록되지 않습니다.
- **목표**: 접수 에이전트가 장애 문의를 기술 에이전트에게 넘기고, 기술 에이전트가 조치 안내 뒤 보상 에이전트에게 넘겨, 같은 대화 이력 위에서 보상 에이전트가 답하는 처리 흐름을 만듭니다. 에이전트가 넘어갈 때마다 이동 경로를 상태에 남깁니다.
    - 에이전트 셋: 접수(가입·요금 문의), 기술(장애 조치 한 문장), 보상(요금 감면 두 문장, 도구 없음).
    - 핸드오프 도구 둘: to_tech(접수→기술), to_refund(기술→보상) — 둘 다 `Command`로 `active_agent`를 바꾸고 경로를 덧붙입니다.
    - 이동 경로: 상태의 `route` 리스트 키(`operator.add` 리듀서).
- **목표 달성 여부의 판정 기준**: 장애와 보상을 함께 말하는 문의를 접수 에이전트에게 넣었을 때, 한 문의가 접수 담당, 기술 담당, 보상 담당 순서로 세 번 응대되고, 이동 경로 두 건(접수에서 기술로, 기술에서 보상으로)이 그 순서대로 상태에 남는 것을 실행 결과에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec06_ex09_s3_diagram.svg)

## 4. 단계별 요구사항

1. **상태를 정의합니다.**
    - 대화 이력(`messages`), 현재 에이전트(`active_agent`), 이동 경로(`route`) 키 세 개를 가지는 상태를 선언합니다.
    - `messages`에는 `add_messages` 리듀서를, `route`에는 리스트를 이어 붙이는 `operator.add` 리듀서를 걸어 도구가 덧붙인 항목이 덮이지 않게 합니다.
2. **핸드오프 도구 두 개를 만듭니다.**
    - to_tech 도구는 `Command`로 「기술 에이전트에게 넘겼습니다」 `ToolMessage`를 남기고 `active_agent`를 기술로 바꾸며 `route`에 「접수→기술」을 덧붙입니다.
    - to_refund 도구는 같은 방식으로 `active_agent`를 보상으로 바꾸며 `route`에 「기술→보상」을 덧붙입니다.
3. **에이전트별 지침과 agent 노드를 만듭니다.**
    - 접수 에이전트는 가입·요금 문의만 다루고 접속 장애 이야기가 나오면 to_tech로 넘깁니다.
    - 기술 에이전트는 장애 조치를 한 문장으로 안내하되, 장애가 이틀 이상 이어졌거나 고객이 보상을 말하면 to_refund로 넘깁니다.
    - 보상 에이전트는 장애 기간에 따른 요금 감면을 두 문장으로 안내하며 도구가 없습니다.
    - agent 노드는 상태의 `active_agent` 값으로 그 에이전트의 시스템 프롬프트와 도구 목록을 고르고, 도구가 있으면 `bind_tools`로 묶은 모델을, 없으면 모델을 그대로 불러 대화 이력에 답을 덧붙입니다.
    - 에이전트가 셋이어도 agent 노드는 하나입니다.
4. **그래프에 노드를 등록합니다.**
    - agent 노드와, 핸드오프 도구를 실행하는 tools 노드(`ToolNode`)를 등록합니다.
5. **엣지를 연결합니다.**
    - START에서 agent로 가는 고정 엣지를 추가합니다.
    - agent 뒤에는 도구 호출이 있으면 tools로, 없으면 END로 가는 조건부 엣지를 추가합니다.
    - tools 뒤에는 agent로 돌아가는 고정 엣지를 추가합니다.
6. **그래프를 컴파일하고 실행합니다.**
    - 장애와 보상을 함께 말하는 문의를 대화 이력에 넣고 시작 에이전트를 접수, 경로를 빈 리스트로 두어 실행한 뒤, 최종 에이전트 값과 이동 경로와 마지막 답을 출력합니다.
    - 값(`ASK`, 고객 문의)은 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
    - agent 노드는 진입할 때 「[agent] 진입 active_agent=값 tool_calls=수」 줄을, 핸드오프 도구는 실행될 때 「[tool] 도구 이름」 줄을 출력합니다.

## 5. 코드 골격 — LangGraph 5단

랭그래프(LangGraph)로 그래프를 세우는 순서는 다음 다섯 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 다섯 단계와 하나씩 대응합니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 상태 정의 | 대화 이력·현재 에이전트·이동 경로의 키를 선언합니다 | `class SupportState(TypedDict)`, `Annotated[list, operator.add]` | 1 |
| ② 노드 함수 정의 | 핸드오프 도구 둘과, 에이전트에 따라 달라지는 agent 함수를 만듭니다 | `@tool`, `Command(update=...)`, `def agent(state) -> dict` | 2, 3 |
| ③ 그래프 빌더 생성과 노드 등록 | 빈 그래프를 열고 agent와 도구 실행 노드를 등록합니다 | `StateGraph(SupportState)`, `add_node`, `ToolNode` | 4 |
| ④ 엣지 연결 | 도구 호출 여부로 나뉘고 도구에서 agent로 돌아오는 분기를 지정합니다 | `add_edge`, `add_conditional_edges` | 5 |
| ⑤ 컴파일과 실행 | 연결을 확정하고 대화 이력과 시작 에이전트를 넣어 실행합니다 | `compile()`, `invoke()` | 6 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import operator
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")
print("모델 준비를 마쳤습니다.")

# 주어진 자료: 고객 문의 ASK — 값을 그대로 씁니다
ASK = ("사흘째 집 인터넷이 끊겨서 재택근무를 못 하고 있습니다. "
       "조치를 받고 싶고, 끊긴 기간만큼 보상도 받고 싶습니다.")

### 단계 ① — 상태 정의 (요구사항 1)

그래프가 도는 동안 모든 노드가 함께 읽고 쓰는 키를 선언합니다. `messages`는 `add_messages` 리듀서가, `route`는 `operator.add` 리듀서가 새 항목을 뒤에 이어 붙입니다. 리듀서가 없으면 도구가 돌려준 `route` 한 항목이 이전 경로를 덮어 버립니다. `active_agent`는 지금 누가 답하는지를 들고 있는 상태 변수입니다.

In [ ]:
# 여기에 단계 ①(상태 정의)을 작성합니다.

### 단계 ② — 노드 함수 정의 (요구사항 2, 3)

- 핸드오프 도구 두 개는 모두 `Command(update=...)`를 돌려줍니다. 각 도구가 바꾸는 값은 `active_agent`와 `route` 키 두 개며, `ToolMessage`에는 `runtime: ToolRuntime` 인자로 받은 `runtime.tool_call_id`를 `tool_call_id=`로 붙입니다.
- 에이전트별 시스템 프롬프트와 도구 목록은 딕셔너리 두 개에 모아 둡니다. 에이전트가 셋이므로 항목도 셋입니다. 접수는 to_tech를, 기술은 to_refund를 들고 있고, 보상은 도구가 없습니다.
- agent 노드는 하나의 함수입니다. 에이전트가 셋으로 늘어도 agent 노드의 코드는 바뀌지 않고, 지침 딕셔너리와 도구 딕셔너리의 항목만 늘어납니다.

In [ ]:
# 여기에 단계 ②(핸드오프 도구 두 개, 에이전트별 지침·도구 목록, agent 노드 정의)를 작성합니다.

### 단계 ③ — 그래프 빌더 생성과 노드 등록 (요구사항 4)

`StateGraph`에 상태를 넘겨 빈 그래프를 열고, `add_node`로 함수마다 이름을 붙여 등록합니다. tools 노드는 함수 대신 `ToolNode`에 핸드오프 도구 두 개의 목록을 넣어 등록합니다. `ToolNode`는 마지막 메시지에 실린 도구 호출을 찾아 도구를 실행하고, 도구가 돌려준 `Command`의 갱신을 상태에 적용합니다.

In [ ]:
# 여기에 단계 ③(그래프 빌더 생성과 노드 등록)을 작성합니다.

### 단계 ④ — 엣지 연결 (요구사항 5)

`add_edge`는 고정된 순서로 연결합니다. `add_conditional_edges`는 판단 함수가 돌려준 이름으로 다음 노드가 나뉘는 분기를 추가합니다. 판단 함수 `need_tool`은 마지막 메시지에 도구 호출이 실려 있으면 tools, 없으면 END를 돌려줍니다. tools 뒤에는 agent로 돌아가는 고정 엣지를 추가합니다. 도구가 갱신한 에이전트 값을 다음 agent 진입이 읽으므로 에이전트를 바꾸는 전용 노드는 필요하지 않습니다.

In [ ]:
# 여기에 단계 ④(판단 함수와 엣지 연결)를 작성합니다.

### 단계 ⑤ — 컴파일과 실행 (요구사항 6)

`compile()`이 연결을 확정해 실행 가능한 그래프를 돌려줍니다. `invoke`에 대화 이력과 시작 에이전트와 빈 경로를 넣으면 최종 상태가 돌아옵니다. 아래에서는 장애와 보상을 함께 말하는 문의를 접수 에이전트에게 넣습니다.

In [ ]:
# 여기에 단계 ⑤(컴파일과 실행)를 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. `[agent] 진입` 줄이 세 번 출력됩니다. `active_agent`가 접수, 기술, 보상 순서로 바뀌고, 앞의 두 번은 `tool_calls=1`, 마지막은 `tool_calls=0`입니다.
2. 진입 줄 사이에 `[tool] to_tech`와 `[tool] to_refund` 줄이 차례로 출력됩니다.
3. 최종 상태의 `route`는 `['접수→기술', '기술→보상']` 두 항목이고, 마지막 답은 보상 에이전트의 요금 감면 안내입니다.

세 가지가 모두 확인되면 완성입니다. 하나라도 다르면 해당 단계의 코드를 다시 봅니다. `route`에 항목이 하나만 남으면 단계 ①의 `route` 리듀서를, 기술 에이전트에서 멈추면 기술 에이전트의 도구 목록과 시스템 프롬프트를 다시 봅니다.